# End-to-End MLOps Pipeline with Kubeflow on GCP

This notebook demonstrates a complete machine learning workflow using Google Cloud Platform (GCP) and Kubeflow Pipelines (KFP). It covers:
1. Installing ML libraries and dependencies
2. Loading and exploring bike-sharing data
3. Building a basic ML model (quick test)
4. Creating reusable Kubeflow pipeline components
5. Orchestrating the ML workflow as an automated pipeline

Let's start by setting up the environment!

## Cell 1: Install Required Libraries

**What this does:**
- Installs Python packages needed for machine learning, cloud integration, and pipeline orchestration
- `--user` flag installs packages for the current user only (best practice in shared environments)
- `--upgrade` ensures we get the latest compatible versions

**Key packages:**
- `scikit-learn`: Machine learning library for building models
- `pandas`: Data manipulation and analysis
- `google-cloud-aiplatform`: Google Cloud AI services
- `kfp`: Kubeflow Pipelines - for orchestrating ML workflows
- `gcsfs`: Access Google Cloud Storage from Python

In [ ]:
USER_FLAG = "--user"

%pip install {USER_FLAG} \
    "scipy" \
    "scikit-learn==1.3.2" \
    "pandas<3.0.0" \
    "google-cloud-aiplatform" \
    "kfp>=2.7.0" \
    "google-cloud-pipeline-components" \
    "gcsfs" \
    "shapely>2" \
    --upgrade

## Cell 2: Restart Jupyter Kernel

**What this does:**
- Restarts the Python kernel after installing new packages
- This ensures all newly installed libraries are loaded into memory
- Similar to restarting your computer after installing software

**Why needed:**
- Some packages (especially C extensions) require a fresh Python interpreter to load properly

In [ ]:
# Cell 2: Restart Kernel
import os
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## Cell 3: Verify Installed Versions

**What this does:**
- Checks the versions of KFP and Google Cloud Pipeline Components installed
- `!python3 -c` runs a Python command directly from the terminal
- `print()` displays the version information

**Why needed:**
- Confirms that the installation was successful
- Version information is important for debugging compatibility issues

**Output explanation:**
- Shows KFP SDK version: 2.17.0
- Shows google_cloud_pipeline_components version: 2.22.0

In [ ]:
!python3 -c "import kfp; print('KFP SDK version: {}'.format(kfp.__version__))"
!python3 -c "import google_cloud_pipeline_components; print('google_cloud_pipeline_components version: {}'.format(google_cloud_pipeline_components.__version__))"

## Cell 4: Retrieve GCP Project ID

**What this does:**
- Fetches your GCP project ID using gcloud CLI
- `!gcloud config list` is a terminal command that reads GCP configuration
- The result is stored in the `PROJECT_ID` variable

**Key concepts:**
- `os.getenv("IS_TESTING")`: Checks if we're in a testing environment
- `shell_output[0]`: Gets the first line from the command output

**Why needed:**
- We need the project ID to access GCP resources (buckets, AI Platform, etc.)
- Storing it in a variable makes it reusable throughout the notebook

In [ ]:
import os
PROJECT_ID = ""
if not os.getenv("IS_TESTING"):
    shell_output=!gcloud config list --format 'value(core.project)' 2>/dev/null
    PROJECT_ID = shell_output[0]
    print("Project ID: ", PROJECT_ID)

## Cell 5: Create GCS Bucket Name

**What this does:**
- Constructs the URL for a Google Cloud Storage (GCS) bucket
- `gs://` is the GCS URL protocol
- Bucket naming: `{PROJECT_ID}-bucket` (e.g., `my-project-bucket`)

**Key concepts:**
- GCS buckets are like cloud folders where we store data, models, and artifacts
- This bucket will hold:
  - Raw dataset
  - Trained ML model
  - Pipeline artifacts

**Example:**
- If PROJECT_ID = "my-gcp-project"
- Then BUCKET_NAME = "gs://my-gcp-project-bucket"

In [ ]:
BUCKET_NAME="gs://" + PROJECT_ID + "-bucket"

## Cell 6: Download Bike-Sharing Dataset

**What this does:**
- Downloads a public dataset from Google Cloud Storage
- `wget` is a command-line tool to download files from the internet
- The file is saved locally as `Bike-Sharing-Dataset.zip`

**Dataset info:**
- Contains bike-sharing demand data
- Includes hourly data with features like temperature, weather, day of week
- We'll predict bike rental demand using this data

**Why this dataset?**
- Real-world data with temporal patterns
- Good for teaching MLOps workflows

In [ ]:
!wget https://storage.googleapis.com/partner-usecase-bucket/ucase009/Bike-Sharing-Dataset.zip

## Cell 7: Extract Dataset

**What this does:**
- Unzips the downloaded file
- `-q` flag means "quiet" (no verbose output)
- `-d data` extracts files into a folder named `data`

**Result:**
- Creates a `data` folder containing CSV files
- `hour.csv` contains the hourly bike-sharing data we'll use

In [ ]:
!unzip -q Bike-Sharing-Dataset.zip -d data

## Cell 8: Upload Data to GCS

**What this does:**
- Copies the CSV file from local computer to Google Cloud Storage bucket
- `gsutil cp` = Google Storage utility copy command
- `./data/hour.csv` = local file path
- `$BUCKET_NAME` = destination cloud folder

**Why needed:**
- The Kubeflow pipeline runs on cloud servers that need to access data
- Cloud storage provides centralized, scalable data access
- Later pipeline components will read from this location

In [ ]:
! gsutil cp ./data/hour.csv $BUCKET_NAME

## Cell 9: Verify Data Upload

**What this does:**
- Lists files in the GCS bucket with size and metadata
- `-al` flags: `a` = all details, `l` = long format (human-readable)
- Confirms the file was uploaded successfully

**Output interpretation:**
- `1156736 bytes` = file size (~1.1 MB)
- `2026-07-13T16:52:50Z` = upload timestamp
- Shows successful upload to cloud

In [ ]:
! gsutil ls -al $BUCKET_NAME

## Cell 10: Import Required Libraries

**What this does:**
- Imports Python libraries needed for ML model building and cloud integration

**Key imports explained:**
- `pickle`: Saves/loads Python objects (for saving trained models)
- `sklearn`: Machine learning library (RandomForestRegressor, metrics)
- `pandas`: Data manipulation (pd.read_csv, dataframes)
- `numpy`: Numerical computing
- `google.cloud.storage`: Access GCS buckets
- `google.cloud.aiplatform`: Google Cloud AI services

**Why this matters:**
- These libraries are the foundation for our ML workflow
- Each one has a specific job (data → modeling → cloud deployment)

In [ ]:
import os
import pprint as pp
import sys

import pickle
import os
import argparse

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error,mean_absolute_percentage_error
import numpy as np
import pandas as pd

from google.cloud import storage
from google.cloud import aiplatform

## Cell 11: Load Data from GCS

**What this does:**
- Reads the CSV file directly from cloud storage using Pandas
- Pandas can work with `gs://` URLs directly (thanks to gcsfs package)
- `.head()` displays the first 5 rows

**Data columns explained:**
- `season`, `yr`, `mnth`, `hr`: Time-based features
- `temp`, `atemp`, `hum`, `windspeed`: Weather features
- `casual`, `registered`: User types
- `cnt`: **Target** - bike rental count (what we want to predict)

**What we see:**
- Data shows hourly bike-sharing counts
- Values are normalized (0.0 to 1.0 for some features)
- Each row = one hour of data

In [ ]:
url = BUCKET_NAME+"/hour.csv"
data = pd.read_csv(url)
data.head()

## Cell 12: Select Modeling Features

**What this does:**
- Selects only the columns we'll use for the ML model
- Removes unnecessary columns (like `instant` and `dteday`)
- Creates a cleaner dataset for training

**Feature selection strategy:**
- We keep 15 columns: 14 features (inputs) + 1 target (`cnt`)
- `instant` (just an ID) and `dteday` (date string) are not useful for the model
- Numeric features are easier for algorithms to work with

In [ ]:
modelling_columns = ["season" 
                     , "yr" 
                     ,"mnth" 
                     ,"hr" 
                     ,"holiday" 
                     , "weekday" 
                     , "workingday" 
                     , "weathersit" 
                     , "temp" 
                     , "atemp" 
                     , "hum" 
                     , "windspeed" 
                     , "casual" 
                     , "registered" 
                     , "cnt"
                    ]
data = data[modelling_columns]

## Cell 13: Split Data into Train/Validation/Test Sets

**What this does:**
- Divides data into 3 parts for proper model evaluation
- 80% training, 10% validation, 10% testing
- `random_state=42` ensures reproducibility (same split every time)
- `sample(frac=1)` shuffles the data randomly

**Why three datasets?**
- **Training set**: Used to teach the model patterns
- **Validation set**: Used to tune hyperparameters during training
- **Test set**: Final evaluation on completely unseen data

**Important principle:**
- Never evaluate on training data (model memorizes, not generalizes)
- Test set simulates real-world performance

In [ ]:
train_size = 0.8
test_size = 0.1
valid_size = 0.1

train_ds, valid_ds, test_ds = np.split(data.sample(frac=1, random_state=42), [int((train_size)*len(data)), int((1-test_size)*len(data))])

## Cell 14: Prepare Features (X) and Target (y)

**What this does:**
- Separates input features (X) from target variable (y) for each dataset
- `drop(columns=target)` removes the "cnt" column from features
- Creates 6 variables total: x_train, y_train, x_valid, y_valid, x_test, y_test

**ML terminology:**
- **X (features)**: Input data (14 columns - temperature, weather, etc.)
- **y (target)**: Output we want to predict (1 column - bike count)

**Why separate?**
- Models need features as input and target as output
- Standard format for all sklearn models

In [ ]:
target = "cnt"

x_train = train_ds.drop(columns=target, axis=1)
y_train = train_ds[target]

x_valid = valid_ds.drop(columns=target, axis=1)
y_valid = valid_ds[target]

x_test = test_ds.drop(columns=target, axis=1)
y_test = test_ds[target]

## Cell 15: Train Random Forest Model

**What this does:**
- Creates and trains a Random Forest machine learning model
- `RandomForestRegressor()`: Algorithm that builds many decision trees and averages their predictions
- `fit(x_train, y_train)`: Trains the model on training data

**What is Random Forest?**
- An ensemble method (combines many weak learners into one strong learner)
- Each tree learns different patterns in the data
- Averaging predictions reduces overfitting (model memorizing)
- Great for both regression (predicting numbers) and classification

**Why Random Forest?**
- Works well with tabular data like ours
- Handles non-linear relationships
- Less parameter tuning needed vs. neural networks

In [ ]:
model = RandomForestRegressor()
model.fit(x_train , y_train)

## Cell 16: Evaluate Model Performance

**What this does:**
- Tests the trained model on validation data
- Calculates multiple evaluation metrics
- Prints results to show how well the model performs

**Metrics explained:**
- **R² (Coefficient of Determination)**: 0.9997 = 99.97% of variance explained (excellent!)
  - Closer to 1.0 = better fit
- **MAE (Mean Absolute Error)**: Average difference between predicted and actual values
  - Lower is better
- **MAPE (Mean Absolute Percentage Error)**: Error as percentage of actual value (0.47% = very good)
- **MSE (Mean Squared Error)**: Average of squared errors (penalizes large errors more)
- **RMSE (Root Mean Squared Error)**: Square root of MSE (same units as target)

**Interpretation:**
- Model is performing very well (R² = 0.9997)
- On average, predictions are off by ~1 bike

In [ ]:
y_pred = model.predict(x_valid)

#evaluate Model
adj_r2 = r2_score(y_true=y_valid, y_pred=y_pred)
mae = mean_absolute_error(y_true=y_valid, y_pred=y_pred)
mse = mean_squared_error(y_true=y_valid, y_pred=y_pred)
mape = mean_absolute_percentage_error(y_true=y_valid, y_pred=y_pred)
rmse = np.sqrt(mse)
print(f"Adjusted R2 : {adj_r2}")
print(f"Mean Absolute Error : {mae}")
print(f"Mean Absolute Percentage Error : {round(mape,4)*100}%")
print(f"Mean Squared Error : {mse}")
print(f"Root Mean Squared Error : {rmse}")

## Cell 17: Save Model and Upload to GCS

**What this does:**
- Saves the trained model to a local pickle file
- Uploads the model file to Google Cloud Storage

**Why pickle?**
- Python's native format for serializing objects
- Preserves the exact model state (weights, hyperparameters, etc.)
- Can be loaded later to make predictions

**Why GCS?**
- Makes model accessible to cloud deployment services
- Can be shared across different environments
- Reliable cloud storage with versioning

**File size:**
- 69.9 MB seems large for one model, but includes all tree data

In [ ]:
MODEL_PATH=BUCKET_NAME+"/models/"
model_path = "./" + "model.pkl"
with open(model_path, 'wb') as file:  
    pickle.dump(model, file) 
    
#copy model artifacts to GCS storage
!gsutil cp "model.pkl" $MODEL_PATH

## Cell 18: Upload Model to Vertex AI Model Registry

**What this does:**
- Registers the trained model in Google Cloud's AI Platform (Vertex AI)
- Specifies the serving container (pre-built Docker image for scikit-learn)
- Creates a versioned model resource in the cloud

**Key parameters:**
- `display_name`: Friendly name for the model
- `artifact_uri`: Path to model files in GCS
- `serving_container_image_uri`: Pre-built Docker image for serving scikit-learn models

**Why Vertex AI?**
- Managed service for ML model deployment
- Handles scaling, monitoring, and versioning
- One resource ID for entire model lifecycle

**Output:**
- Gives us a `resource_name` to reference the model later

In [ ]:
#Prediction containers list available at : https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers
serving_container_uri = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"

#define GCS location for model artifacts
artifact_uri = MODEL_PATH

#Upload Model to Vertex AI Model Registry using Python SDK
model = aiplatform.Model.upload(display_name= "MLOps0-model" ,
                                    artifact_uri=artifact_uri,
                                    serving_container_image_uri=serving_container_uri)

## Cell 19: Deploy Model to Endpoint

**What this does:**
- Creates a cloud endpoint (server) that serves predictions
- Makes the model accessible via REST API
- Configures compute resources (n1-standard-4 machine type)
- Sets auto-scaling: 1-1 replicas (1 instance at all times)

**What is an Endpoint?**
- A cloud service that exposes your model via HTTP API
- Clients send data → endpoint → model → predictions returned
- Like a restaurant: orders come in, kitchen processes, food goes out

**Machine type:**
- `n1-standard-4`: 4 vCPU, good for inference (predictions)
- Other options: n1-standard-2 (smaller), n1-highmem-8 (more memory)

**Scaling:**
- `min_replica_count=1`: Always have 1 instance running
- `max_replica_count=1`: Never scale beyond 1 (cost-saving for demo)

In [ ]:
#Create the model endpoint using Python SDK
endpoint = model.deploy(machine_type="n1-standard-4",
                        min_replica_count=1,
                        max_replica_count=1)

## Cell 20: Test Model Endpoint

**What this does:**
- Sends test data to the deployed endpoint
- Receives predictions from the cloud model
- Validates that the serving pipeline works end-to-end

**Request format:**
- `instances`: List of input data points
- Each item is a list of 14 feature values (matching model training)
- Example: `[season=1, yr=0, mnth=1, hr=0, ...]`

**What happens:**
1. Client sends HTTP request with instance data
2. Endpoint receives request
3. Model runs inference (prediction)
4. Prediction returned to client

**Real-world analogy:**
- Like calling a restaurant hotline: you provide order details, they confirm timing

In [ ]:
#Test the model endpoint using Python SDK

#create list to hold request data
instances = [
    [1.0, 0.0, 1.0, 0.0, 0.0, 6.0, 0.0, 1.0, 0.24, 0.2879, 0.81, 0.0, 3.0, 13.0],
  ]

prediction = endpoint.predict(instances=instances)

print(prediction)

## Cell 21: Clean Up Resources

**What this does:**
- Undeploys the model (stops the endpoint)
- Deletes the endpoint (removes cloud resources)
- Deletes the model from registry

**Why clean up?**
- Endpoints and models consume cloud resources (costs money)
- Good practice to clean up after testing
- Makes room for new deployments

**Cost impact:**
- Running endpoints cost ~$10-50 per month
- Proper cleanup prevents surprise bills

In [ ]:
# Undeploy the model and delete the endpoint
endpoint.undeploy_all()
endpoint.delete()
model.delete()

---

# Part 2: Building Kubeflow Pipeline Components

Now we transition from manual ML workflow to **automated, reproducible MLOps**.

## Key Concepts:
- **Components**: Reusable, containerized ML tasks (download, preprocess, train, evaluate)
- **Pipeline**: DAG (directed acyclic graph) connecting components
- **Orchestration**: Automatic scheduling, error handling, resource management

Instead of running cells manually, Kubeflow automates:
- Running components in the right order
- Passing data between components
- Scaling to handle large datasets
- Tracking artifacts and metrics

## Cell 22: Import Kubeflow and Component Libraries

**What this does:**
- Imports KFP libraries for building components and pipelines
- Imports data type definitions (Dataset, Model, Metrics)

**Key imports explained:**
- `@component`: Decorator to convert Python functions into KFP components
- `dsl`: Domain-Specific Language for defining pipelines
- `Input/Output`: Type hints for component I/O
- `Dataset/Model`: Special types for ML artifacts
- `Metrics`: For logging model evaluation results
- `compiler`: Converts Python pipeline definition to Kubernetes YAML

**Why these imports?**
- `@component` turns Python functions into cloud tasks
- `dsl.pipeline` creates the entire workflow graph
- Type hints enable data validation and component composition

In [ ]:
from typing import NamedTuple

import kfp
from kfp import dsl
from kfp.dsl import (Artifact, Dataset, Input, InputPath, Model, Output,
                        OutputPath, component , ClassificationMetrics , Metrics)

from kfp import compiler
from kfp.components import load_component_from_file

import json
import yaml

## Cell 23: Create Components Directory

**What this does:**
- Creates a `components` folder to store component definitions
- Each component will be saved as a separate YAML file

**Why separate files?**
- Modularity: Each component is independent
- Reusability: Components can be used in different pipelines
- Version control: Easy to track changes to individual components

In [ ]:
!mkdir components

---

# Component 1: Download Data

## Cell 24: Define Download Component

**What this does:**
- Creates a reusable component that downloads data from cloud storage
- Uses `@component` decorator to define it as a KFP component

**Function parameters:**
- `input_data_path`: GCS bucket path (e.g., "gs://my-bucket")
- `input_data_filename`: File name (e.g., "hour.csv")
- `downloaded_data`: Output dataset to be passed to next component

**Component decorator options:**
- `packages_to_install`: Python packages needed (pandas, gcsfs, fsspec)
- `base_image`: Docker image to run component in (python:3.9)
- `output_component_file`: YAML file path for this component

**What the function does:**
1. Joins input path + filename into full URL
2. Reads CSV from cloud storage using pandas
3. Writes to output dataset path (for next component)

**Key Kubeflow concepts:**
- `Input[Dataset]`: Read-only input artifact
- `Output[Dataset]`: Writable output artifact
- Artifacts are files stored between pipeline steps

In [ ]:
@component(
    packages_to_install=["pandas", "pyarrow" , "fsspec" , "gcsfs"],
    base_image="python:3.9",
    output_component_file="./components/download_data.yaml"
)
def download_data(input_data_path : str
                  , input_data_filename : str
                  , downloaded_data : Output[Dataset]):
    
    import pandas as pd
    import os
    
    print(f"input_data_path : {input_data_path}")
    print(f"input_data_filename : {input_data_filename}")
    print(f"downloaded_data : {downloaded_data}")
    print(f"downloaded_data.path : {downloaded_data.path}")
                
    url = os.path.join(input_data_path , input_data_filename)
    
    #read data from GCS location
    data = pd.read_csv(url)
    
    #write to output dataset path
    output_data_uri = downloaded_data.path + ".csv" 
    data.to_csv(output_data_uri 
                , index=False
                , encoding='utf-8-sig')

---

# Component 2: Preprocess Data

## Cell 25: Define Preprocessing Component

**What this does:**
- Takes downloaded data and splits it into train/validation/test sets
- Selects relevant features
- Outputs 3 datasets for subsequent components

**Function parameters:**
- Inputs:
  - `train_size, test_size, valid_size`: Split ratios (0.8, 0.1, 0.1)
  - `input_data`: Dataset from download component
- Outputs:
  - `train_data, valid_data, test_data`: Three separate datasets

**What the function does:**
1. Reads input CSV
2. Selects 15 important columns
3. Shuffles data randomly
4. Splits into 3 parts using indices
5. Writes each split to separate CSV file

**Data flow:**
```
Input Data (1 CSV)
    ↓
  Select Columns
    ↓
  Shuffle & Split
    ↓
Output: 3 CSVs (train, valid, test)
```

In [ ]:
@component(
    packages_to_install=["pandas", "pyarrow", "fsspec" , "gcsfs"],
    base_image="python:3.9",
    output_component_file="./components/preprocess_data.yaml"
)
def preprocess_data(train_size : float 
                    , test_size : float
                    , valid_size : float
                    , train_data : Output[Dataset]
                    , valid_data : Output[Dataset]
                    , test_data : Output[Dataset]
                    , input_data:  Input[Dataset]
                   ):
    
    import numpy as np
    import pandas as pd
    
    print(f"train_size : {train_size}")
    print(f"test_size : {test_size}")
    print(f"valid_size : {valid_size}")
    print(f"train_data : {train_data}")
    print(f"valid_data : {valid_data}")
    print(f"test_data : {test_data}")
    print(f"input_data : {input_data}")
    
    data = pd.read_csv(input_data.path + ".csv")
    
    modelling_columns = ["season" 
                     , "yr" 
                     ,"mnth" 
                     ,"hr" 
                     ,"holiday" 
                     , "weekday" 
                     , "workingday" 
                     , "weathersit" 
                     , "temp" 
                     , "atemp" 
                     , "hum" 
                     , "windspeed" 
                     , "casual" 
                     , "registered" 
                     , "cnt"
                    ]
    
    data = data[modelling_columns]

    train_ds, valid_ds, test_ds = np.split(data.sample(frac=1, random_state=42), [int((train_size)*len(data)), int((1-test_size)*len(data))])
    train_ds.to_csv(train_data.path + ".csv" , index=False, encoding='utf-8-sig')
    valid_ds.to_csv(valid_data.path + ".csv" , index=False, encoding='utf-8-sig')
    test_ds.to_csv(test_data.path + ".csv" , index=False, encoding='utf-8-sig')

---

# Component 3: Train Model

## Cell 26: Define Training Component

**What this does:**
- Takes training data and trains a Random Forest model
- Saves the trained model as pickle file
- Stores model metadata (name, framework, version)

**Function parameters:**
- Inputs:
  - `train_data`: Training dataset from preprocess component
- Outputs:
  - `model`: Trained model artifact

**What the function does:**
1. Reads training CSV
2. Creates RandomForestRegressor with default hyperparameters
3. Separates features (X) and target (y)
4. Trains model using `fit(x_train, y_train)`
5. Adds metadata (model name, framework, version)
6. Serializes model to pickle file

**Key differences from manual workflow:**
- Runs in isolated container environment
- Only uses training data (no manual train/valid split needed)
- Outputs artifact that flows to next component

**Output:**
- `model.pkl` file in GCS for later deployment

In [ ]:
@component(
    packages_to_install=["kfp==2.4.0", "pandas", "pyarrow",  "scikit-learn==1.3.2" , "fsspec" , "gcsfs", "click==8.1.7", "docstring-parser==0.16", "kfp-pipeline-spec==0.2.2", "kfp-server-api==2.0.5", "kubernetes==26.1.0", "PyYAML==6.0.2", "requests-toolbelt==0.10.1", "tabulate==0.9.0", "protobuf==3.20.3", "urllib3==1.26.20"]
    , base_image="python:3.9"
    , output_component_file="./components/train.yaml"
)
def train_model(
    train_data:  Input[Dataset],
    model: Output[Model], 
):
    
    print(f"train_data : {train_data}")
    print(f"model : {model}")
    
    from sklearn.ensemble import RandomForestRegressor
    import pandas as pd
    import pickle
    import sklearn

    train_ds = pd.read_csv(train_data.path+".csv")
    my_model = RandomForestRegressor()
    
    target = "cnt"
    
    x_train = train_ds.drop(columns=target, axis=1)
    y_train = train_ds[target]
    
    my_model.fit(x_train , y_train)
    model.metadata["model_name"] = "RandomForestRegressor"
    model.metadata["framework"] = "sklearn"
    model.metadata["framework_version"] = sklearn.__version__
    file_name = model.path + f".pkl"
    
    with open(file_name, 'wb') as file:  
        pickle.dump(my_model, file)

---

# Component 4: Evaluate Model

## Cell 27: Define Evaluation Component

**What this does:**
- Tests the trained model on test dataset
- Calculates performance metrics
- Decides whether model is good enough to deploy (returns True/False flag)

**Function parameters:**
- Inputs:
  - `test_data`: Test dataset from preprocess component
  - `model`: Trained model from training component
  - `target_column_name`: Name of target variable ("cnt")
  - `deployment_metric`: Metric to use for deployment decision ("r2")
  - `deployment_metric_threshold`: Minimum acceptable metric value (0.8)
- Outputs:
  - `kpi`: Metrics object to log all evaluation results
  - Return value: `deploy_flag` ("True" or "False")

**What the function does:**
1. Loads trained model from pickle
2. Reads test dataset
3. Makes predictions on test data
4. Calculates 5 evaluation metrics
5. Logs metrics to KFP (visible in UI dashboard)
6. Compares deployment_metric to threshold
7. Returns "True" if metric >= threshold, else "False"

**Conditional deployment:**
- If R² >= 0.8: Model is "good enough" → mark for deployment
- If R² < 0.8: Model needs improvement → skip deployment

**Metrics logged:**
- R², MAE, MAPE, MSE, RMSE
- All visible in Kubeflow UI for tracking

In [ ]:
@component(
    packages_to_install=["pandas", "pyarrow",  "scikit-learn==1.3.2" , "fsspec" , "gcsfs"]
    , base_image="python:3.9"
    , output_component_file="./components/evaluate_model.yaml"
)
def evaluate_model(
    test_data:  Input[Dataset],
    model: Input[Model], 
    target_column_name : str ,
    deployment_metric : str ,
    deployment_metric_threshold : float ,
    kpi: Output[Metrics]
)-> NamedTuple(
    "Outputs",
    [
        ("deploy_flag", str),  # Return parameter.
    ],
):
    
    print(f"test_data : {test_data}")
    print(f"model : {model}")
    print(f"kpi : {kpi}")
    print(f"deployment_metric : {deployment_metric}")
    print(f"deployment_metric_threshold : {deployment_metric_threshold}")
    
    from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error,mean_absolute_percentage_error
    import pandas as pd
    import pickle
    import numpy as np
    import json
    
    test_ds = pd.read_csv(test_data.path+".csv")
    target = target_column_name
    
    x_test = test_ds.drop(columns=target, axis=1)
    y_test = test_ds[target]
    
    print(f"model.path : {model.path}")
    file_name = model.path + f".pkl"
    print(f"file_name : {file_name}")
    #model = pickle.loads(file_name)
    with open(file_name, 'rb') as file:  
        model = pickle.load(file)
    
    y_pred = model.predict(x_test)
    r2 = r2_score(y_true=y_test, y_pred=y_pred)
    mae = mean_absolute_error(y_true=y_test, y_pred=y_pred)
    mse = mean_squared_error(y_true=y_test, y_pred=y_pred)
    mape = mean_absolute_percentage_error(y_true=y_test, y_pred=y_pred)
    rmse = np.sqrt(mse)
    
    model_metrics = {"r2" : r2 
                     , "mae" : mae 
                     , "mape" : mape 
                     , "mse" : mse 
                     , "rmse" : rmse
                    }
    
    print(f"Adjusted_R2 : {r2}")
    print(f"Mean Absolute Error : {mae}")
    print(f"Mean Absolute Percentage Error : {round(mape,4)*100}%")
    print(f"Mean Squared Error : {mse}")
    print(f"Root Mean Squared Error : {rmse}")
    
    kpi.log_metric("Adjusted_R2", float(r2))
    kpi.log_metric("Mean Absolute Error", float(mae))
    kpi.log_metric("Mean Absolute Percentage Error", float(mape))
    kpi.log_metric("Mean Squared Error", float(mse))
    kpi.log_metric("Root Mean Squared Error", float(rmse))
    
    actual_metric_value = model_metrics.get(deployment_metric)
    
    if actual_metric_value >= deployment_metric_threshold:
        deploy_flag = "True"
    else:
        deploy_flag = "False"
        
    return (deploy_flag,)

---

# Component 5: Register Model

## Cell 28: Define Model Registration Component

**What this does:**
- Uploads trained model to Vertex AI Model Registry
- Makes model available for deployment
- Returns model resource name for later deployment

**Function parameters:**
- Inputs:
  - `serving_container_uri`: Docker image for serving sklearn models
  - `project_id`: GCP project ID
  - `region`: GCP region (e.g., "us-central1")
  - `model_name`: Display name for the model
  - `model`: Trained model artifact
- Output:
  - `model_resource_name`: Cloud resource ID (e.g., "projects/123/models/456@1")

**What the function does:**
1. Initializes Vertex AI client
2. Calls `aiplatform.Model.upload()`
3. Specifies model artifact location (GCS path)
4. Specifies serving container image
5. Returns registered model's resource name

**Key concepts:**
- `artifact_uri`: Path to model.pkl file in GCS
- `serving_container_image_uri`: Pre-built Docker image for serving
- Returns resource name that uniquely identifies model in Vertex AI

**Why Vertex AI?**
- Central model registry for entire organization
- Version control built-in
- Ready for production deployment

In [ ]:
@component(
    packages_to_install=["pandas", "pyarrow",  "scikit-learn==1.3.2" , "fsspec" , "gcsfs" , "google-cloud-aiplatform"]
    , base_image="python:3.9"
    , output_component_file="./components/register_model.yaml"
)
def register_model(
    serving_container_uri : str ,
    project_id : str ,
    region: str,
    model_name : str , 
    model: Input[Model], 
)-> NamedTuple(
    "Outputs",
    [
        ("model_resource_name", str),  # Return parameter.
    ],
):
    
    print(f"serving_container_uri : {serving_container_uri}")
    print(f"project_id : {project_id}")
    print(f"region : {region}")
    print(f"model : {model}")
    
    from google.cloud import aiplatform
    
    print(f"model.uri : {model.uri[:-5]}")
    
    aiplatform.init(project = project_id , location=region)
    model = aiplatform.Model.upload(display_name= model_name ,
                                    artifact_uri=model.uri[:-5],
                                    serving_container_image_uri=serving_container_uri)
    return (model.resource_name,)

---

# Component 6: Deploy Model

## Cell 29: Define Model Deployment Component

**What this does:**
- Deploys registered model to a cloud endpoint
- Makes model accessible via REST API for real-time predictions
- Returns endpoint resource name

**Function parameters:**
- Inputs:
  - `model_resource_name`: Resource ID from registration component
  - `project_id`: GCP project ID
  - `region`: GCP region
- Output:
  - `endpoint_resource_name`: Cloud endpoint ID for making predictions

**What the function does:**
1. Initializes Vertex AI client
2. Loads model using its resource name
3. Creates endpoint with specified machine type
4. Deploys model to endpoint (auto-scales 1-1 replicas)
5. Returns endpoint resource name

**Machine configuration:**
- `machine_type="n1-standard-4"`: 4 vCPU, 15GB RAM
- `min_replica_count=1`: Always running
- `max_replica_count=1`: No auto-scaling (cost control)

**What happens next:**
- Endpoint becomes available for real-time predictions
- Clients send data → endpoint serves predictions
- Usage metrics tracked in Vertex AI monitoring

In [ ]:
@component(
    packages_to_install=["kfp==2.4.0", "pandas", "pyarrow",  "scikit-learn==1.3.2" , "fsspec" , "gcsfs", "google-cloud-aiplatform", "click==8.1.7", "kfp-pipeline-spec==0.2.2", "kfp-server-api==2.0.5", "kubernetes==26.1.0", "PyYAML==6.0.2", "requests-toolbelt==0.10.1", "tabulate==0.9.0", "protobuf==3.20.3", "urllib3==1.26.20", "numpy==1.24.4", "google-cloud-pipeline-components==2.8.0"]
    , base_image="python:3.9"
    , output_component_file="./components/deploy_model.yaml"
)
def deploy_model(
    model_resource_name : str ,
    project_id : str ,
    region: str
)-> NamedTuple(
    "Outputs",
    [
        ("endpoint_resource_name", str),  # Return parameter.
    ],
):
    
    print(f"model_resource_name : {model_resource_name}")
    print(f"project_id : {project_id}")
    print(f"region : {region}")
    
    from google.cloud import aiplatform
    
    aiplatform.init(project = project_id , location=region)
    
    model = aiplatform.Model(model_resource_name)
    endpoint = model.deploy(machine_type="n1-standard-4",
                        min_replica_count=1,
                        max_replica_count=1)
    
    return (endpoint.resource_name,)

---

# Part 3: Build and Run the Pipeline

## Cell 30: Get GCP Configuration

**What this does:**
- Retrieves GCP configuration (service account, project ID, bucket)
- Prints values for verification

**Key variables:**
- `SERVICE_ACCOUNT`: Cloud service account for authentication
- `PROJECT_ID`: Your GCP project
- `BUCKET_NAME`: Cloud storage bucket

**Why needed:**
- Pipeline components need credentials to access GCP resources
- Service account provides programmatic access
- Same bucket stores data, models, and artifacts

In [ ]:
shell_output = !gcloud auth list 2>/dev/null
SERVICE_ACCOUNT = shell_output[2].replace("*", "").strip()
print("Service Account:", SERVICE_ACCOUNT)
print("Project ID: ", PROJECT_ID)
print("staging_bucket_uri: ",BUCKET_NAME)
print("input_data_path: ",BUCKET_NAME)

## Cell 31: Create Pipeline Configuration File

**What this does:**
- Creates a JSON configuration file with all pipeline parameters
- Centralizes configuration for easy updates
- Configuration includes data paths, model parameters, and deployment settings

**Config parameters explained:**

**GCP resources:**
- `project`: Your GCP project ID
- `region`: Deployment region (us-central1)
- `service_account`: Account with permissions
- `staging_bucket_uri`: Cloud storage for artifacts

**Pipeline settings:**
- `pipeline_name`: Friendly name for pipeline
- `pipeline_package_path`: Output file (compiled to Kubernetes YAML)

**Data parameters:**
- `input_data_path`, `input_data_filename`: Where to load data
- `train_size, test_size, valid_size`: Data split ratios
- `target_column_name`: What we're predicting ("cnt")

**Model parameters:**
- `deployment_metric`: Metric for go/no-go decision ("r2")
- `deployment_metric_threshold`: Minimum acceptable value (0.8)
- `model_name`: Registry name
- `serving_container_uri`: Docker image for serving

**Why JSON?**
- Easy to version control
- Can be used by multiple tools/scripts
- Human-readable configuration

In [ ]:
%%writefile config.json
{
    "project":"qwiklabs-gcp-01-d9effd0f78c6",
    "region":"us-central1",
    "service_account":"357529305585-compute@developer.gserviceaccount.com",
    "staging_bucket_uri":"gs://qwiklabs-gcp-01-d9effd0f78c6-bucket",
    "pipeline_name":"tabular-data-regression-kfp-cicd-pipeline",
    "pipeline_package_path":"tabular-data-regression-kfp-cicd-pipeline.json",
    "input_data_path":"gs://qwiklabs-gcp-01-d9effd0f78c6-bucket",
    "input_data_filename":"hour.csv",
    "target_column_name":"cnt",
    "train_size":0.8,
    "test_size":0.1,
    "valid_size":0.1,
    "deployment_metric":"r2",
    "deployment_metric_threshold":0.8,
    "model_name":"model_tabular_regression",
    "serving_container_uri":"us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"
}

## Cell 32: Build Pipeline Definition

**What this does:**
- Loads all component definitions from YAML files
- Creates a `@dsl.pipeline` function that orchestrates components
- Compiles pipeline definition to JSON format
- Creates a Kubernetes manifest for cloud execution

**Key sections:**

**1. Load components:**
- Each component was saved as YAML
- `load_component_from_file()` imports them back as functions

**2. Define pipeline:**
- `@dsl.pipeline()`: Decorator that converts function to pipeline
- `pipeline_root`: Where artifacts are stored (GCS path)
- Function parameters: All config values passed through

**3. Component sequencing:**
- `download_data_op`: Gets data
- `preprocess_data_op`: Splits data (uses output from download)
- `train_model_op`: Trains model (uses training data)
- `evaluate_model_op`: Tests model (uses test data + trained model)
- **Conditional block**: `with dsl.If(evaluate_model_op.outputs["deploy_flag"] == "True")`
  - `register_model_op`: Only runs if evaluation passes
  - `deploy_model_op`: Only runs if evaluation passes

**4. Compile:**
- `compiler.Compiler().compile()`: Converts pipeline to JSON
- Output: `tabular-data-regression-kfp-cicd-pipeline.json`

**Data flow visualization:**
```
download_data
    ↓ (downloaded_data)
preprocess_data
    ↓ (train_data, test_data)
    ├→ train_model
    │   ↓ (model)
    │   ├→ evaluate_model
    │   │   ↓ (deploy_flag)
    │   │   if deploy_flag == "True":
    │   │       ↓
    │   │     register_model
    │   │       ↓
    │   │     deploy_model
    │   └→ test_data
    └→ evaluate_model
```

In [ ]:
%%writefile build_pipeline.py

import json
import yaml

import kfp
from kfp import dsl
from kfp import compiler
from kfp.components import load_component_from_file

download_data = load_component_from_file("./components/download_data.yaml")
preprocess_data = load_component_from_file("./components/preprocess_data.yaml")
train_model = load_component_from_file("./components/train.yaml")
evaluate_model = load_component_from_file("./components/evaluate_model.yaml")
register_model = load_component_from_file("./components/register_model.yaml")
deploy_model = load_component_from_file("./components/deploy_model.yaml")

#read configuration from file
with open("config.json") as json_file:
    config = json.load(json_file)
    
PIPELINE_NAME = config.get("pipeline_name")
PACKAGE_PATH = config.get("pipeline_package_path")
BUCKET_URI = config.get("staging_bucket_uri")
PIPELINE_ROOT = "{}/pipeline_root/kfp_tabular_data_regression".format(BUCKET_URI)
print(f"PIPELINE_ROOT :{PIPELINE_ROOT}")

@dsl.pipeline(
    # Default pipeline root. You can override it when submitting the pipeline.
    pipeline_root=PIPELINE_ROOT,
    # A name for the pipeline. Use to define the pipeline Context.
    name=PIPELINE_NAME,
    
)
def pipeline(project: str = "",
 region: str = "",
 service_account: str = "",
 staging_bucket_uri: str = "",
 pipeline_name: str = "",
 pipeline_package_path: str = "",
 input_data_path: str = "",
 input_data_filename: str = "",
 target_column_name: str = "",
 train_size: float = 0.8,
 test_size: float = 0.1,
 valid_size: float = 0.1,
 hypertune_container_image_uri: str = "" ,
 hypertune_machine_type: str = "",
 hypertune_machine_replica_count: int = 1 ,
 hypertune_max_trial_count: int = 1 ,
 hypertune_parallel_trial_count: int = 1 ,
 hypertune_metric : str = "" ,
 hypertune_metric_objective : str = "" ,
 hypertune_job_name: str = "" ,
 deployment_metric: str = "" ,
 deployment_metric_threshold: float = 0.8 ,
 serving_container_uri : str = "" ,
 model_name : str = "", 
 user_email : str = "",
 monitoring_job_name : str = "" ,
 predict_instance_schema_uri : str = ""
):
    
    download_data_op = download_data(input_data_path = input_data_path 
                                     , input_data_filename = input_data_filename
                                    )
    
    
    preprocess_data_op = preprocess_data(train_size = train_size
                                         , test_size = test_size
                                         , valid_size = valid_size
                                         , input_data = download_data_op.outputs["downloaded_data"])
    
    train_model_op = train_model(train_data = preprocess_data_op.outputs["train_data"])
    
    
    evaluate_model_op = evaluate_model(test_data = preprocess_data_op.outputs["test_data"]
                                       ,model = train_model_op.outputs["model"]
                                       ,target_column_name = target_column_name
                                       ,deployment_metric = deployment_metric 
                                       ,deployment_metric_threshold = deployment_metric_threshold
                                      )
    
    with dsl.If(evaluate_model_op.outputs["deploy_flag"] == "True"):
        
        register_model_op = register_model(serving_container_uri = serving_container_uri 
                                       , model = train_model_op.outputs["model"]
                                       , model_name = model_name
                                       , project_id = project 
                                       , region = region)
        
        #deploy only if metric value exceeds deployment threshold
        deploy_model_op = deploy_model(model_resource_name = register_model_op.outputs["model_resource_name"] 
                                   , project_id = project 
                                   , region = region)
    
compiler.Compiler().compile(
    pipeline_func=pipeline
    , package_path=PACKAGE_PATH
)

## Cell 33: Create Pipeline Submission Script

**What this does:**
- Creates a Python script to submit pipeline to Vertex AI Pipelines
- This script can be scheduled or triggered by CI/CD systems

**Key steps:**
1. Load config from JSON
2. Create `PipelineJob` object with:
   - Pipeline name
   - Path to compiled pipeline (JSON)
   - Pipeline root (GCS path for artifacts)
   - All configuration parameters
3. Submit job with service account credentials

**Execution:**
- `job.submit()`: Sends pipeline to Vertex AI
- Kubeflow Pipelines orchestrates execution
- Can be viewed in Vertex AI Pipelines UI

**Typical flow:**
```
python run_pipeline.py
    ↓
Create PipelineJob
    ↓
Submit to Vertex AI
    ↓
Kubeflow schedules component execution
    ↓
Monitor in UI (https://console.cloud.google.com/vertex-ai/pipelines)
```

In [ ]:
%%writefile run_pipeline.py

from google.cloud import aiplatform
import yaml
import json

with open("config.json") as json_file:
    config = json.load(json_file)
    
SERVICE_ACCOUNT = config.get("service_account")
DISPLAY_NAME = config.get("pipeline_name")
PACKAGE_PATH = config.get("pipeline_package_path")
BUCKET_URI = config.get("staging_bucket_uri")
PIPELINE_ROOT = "{}/pipeline_root/kfp_tabular_data_regression".format(BUCKET_URI)
print(f"PIPELINE_ROOT :{PIPELINE_ROOT}")

job = aiplatform.PipelineJob(
    display_name=DISPLAY_NAME,
    template_path=PACKAGE_PATH,
    pipeline_root=PIPELINE_ROOT,
    parameter_values=config,
)

job.submit(service_account = SERVICE_ACCOUNT)

## Cell 34: Execute Pipeline Build Script

**What this does:**
- Runs the build_pipeline.py script
- Compiles all components into a single pipeline JSON file
- Creates the orchestration definition

**Output:**
- `tabular-data-regression-kfp-cicd-pipeline.json`: Complete pipeline definition
- Ready to be submitted to Vertex AI Pipelines

In [ ]:
!python3 build_pipeline.py

## Cell 35: Submit Pipeline to Vertex AI

**What this does:**
- Executes the run_pipeline.py script
- Submits the compiled pipeline to Vertex AI Pipelines
- Starts automated ML workflow execution

**What happens next:**
1. Vertex AI receives pipeline job
2. Kubeflow Pipelines service schedules components
3. Each component runs in its own container
4. Pipeline orchestrates data flow between components
5. Metrics and artifacts tracked throughout
6. Model deployed only if evaluation passes

**Monitoring:**
- Go to: https://console.cloud.google.com/vertex-ai/pipelines
- Click on pipeline name to see real-time execution
- View component logs, metrics, and outputs

In [ ]:
!python3 run_pipeline.py

## Cell 36: Upload Pipeline Artifacts to GCS

**What this does:**
- Uploads configuration and pipeline definition to cloud storage
- Makes them accessible to other services
- Sets public read permissions for sharing

**Files uploaded:**
1. `config.json`: Pipeline parameters
2. `tabular-data-regression-kfp-cicd-pipeline.json`: Pipeline definition

**Access control:**
- `gsutil acl ch -u AllUsers:R`: Makes files readable by all
- Enables sharing and CI/CD integration

**Use cases:**
- CI/CD pipelines can fetch and run
- Teams can share pipeline definitions
- Version control in cloud storage

In [ ]:
! gsutil cp ./config.json $BUCKET_NAME
! gsutil cp ./tabular-data-regression-kfp-cicd-pipeline.json $BUCKET_NAME
! gsutil acl ch -u AllUsers:R $BUCKET_NAME/config.json
! gsutil acl ch -u AllUsers:R $BUCKET_NAME/tabular-data-regression-kfp-cicd-pipeline.json

---

# Summary: Complete MLOps Workflow

## What We Built

This notebook demonstrates a **production-ready ML workflow** with:

### Part 1: Manual ML Development
- Load and explore data
- Train and evaluate model
- Deploy to cloud endpoint
- Make real-time predictions

### Part 2: Kubeflow Components
- **Download**: Fetch data from cloud storage
- **Preprocess**: Split into train/validation/test
- **Train**: Build Random Forest model
- **Evaluate**: Test and generate metrics
- **Register**: Upload model to Vertex AI
- **Deploy**: Create prediction endpoint

### Part 3: Pipeline Orchestration
- Compose components into DAG
- Add conditional deployment logic
- Compile to Kubernetes manifest
- Submit to Vertex AI for execution

## Key MLOps Benefits

| Aspect | Manual | Kubeflow Pipeline |
|--------|--------|-------------------|
| **Reproducibility** | Run cells manually | Automated, versioned |
| **Scaling** | One dataset only | Handles large datasets |
| **Error Handling** | Manual intervention | Automatic retry logic |
| **Monitoring** | Printf debugging | Comprehensive metrics |
| **Scheduling** | Manual trigger | Cron jobs, event-driven |
| **Collaboration** | Share notebook | Version-controlled pipeline |
| **CI/CD** | Not applicable | Integrates with deployment |

## Next Steps

1. **Monitor pipeline execution**: Check Vertex AI Pipelines UI
2. **View model metrics**: Look at logged metrics and artifacts
3. **Test predictions**: Use endpoint resource name to make real-time predictions
4. **Schedule runs**: Set up Cloud Scheduler to run pipeline daily/weekly
5. **Add data validation**: Include data quality checks in components
6. **Implement monitoring**: Track model drift and prediction quality in production
7. **Scale resources**: Adjust machine types and replicas based on load

## Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                     Kubeflow Pipeline                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Data Source (GCS) → Download → Preprocess → {Train, Evaluate} │
│                                                    ↓             │
│                                         Metric Threshold Check   │
│                                                    ↓             │
│                                     Register → Deploy → Endpoint │
│                                                                 │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │ Each component runs in isolated container              │   │
│  │ - Specified Python version                            │   │
│  │ - Custom package dependencies                         │   │
│  │ - Automatic scaling on Kubernetes                     │   │
│  └─────────────────────────────────────────────────────────┘   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
                              ↓
              ┌───────────────────────────────┐
              │  Vertex AI Model Registry     │
              │  (Model versioning & serving) │
              └───────────────────────────────┘
                              ↓
              ┌───────────────────────────────┐
              │  Prediction Endpoint          │
              │  (Real-time inference API)    │
              └───────────────────────────────┘
```

## Key Takeaways

✓ **Modularity**: Each component is reusable and testable independently

✓ **Reproducibility**: Same inputs always produce same outputs

✓ **Scalability**: Handles large datasets and high-volume predictions

✓ **Maintainability**: Easy to update components without breaking others

✓ **Observability**: Metrics and artifacts tracked throughout pipeline

✓ **Production-ready**: Integrates with Google Cloud best practices